# 뉴욕시 Airbnb 숙소 가격 예측 분석

**분석 목표:** 뉴욕시 Airbnb 숙소의 다양한 피처(feature)를 활용하여 숙소 가격(price)을 예측하는 최적의 모델을 구축한다.

**데이터셋:** `AB_NYC_2019.csv`

---

## 1단계: EDA (탐색적 데이터 분석)

### 1.1. 데이터 불러오기 및 기본 정보 확인

- **수행 내용:** Pandas 라이브러리를 사용하여 `AB_NYC_2019.csv` 파일을 데이터프레임으로 불러오고, 데이터의 기본 구조(행/열), 데이터 타입, 결측치 존재 여부를 확인합니다.
- **선택 근거:** 데이터 분석의 가장 첫 단계로, 데이터의 원본 형태와 기본적인 구조를 파악해야 향후 처리 방향을 설정할 수 있습니다.

In [ ]:
import pandas as pd
import numpy as np

# 데이터 불러오기
df = pd.read_csv('AB_NYC_2019.csv')

# 최상위 5개 행 확인
print('--- 데이터 샘플 ---')
print(df.head())

# 데이터 타입 및 Non-null 개수 확인
print('
--- 데이터 정보 ---')
df.info()

# 컬럼별 결측치 개수 확인
print('
--- 결측치 확인 ---')
print(df.isnull().sum())

### 1.2. 기술 통계 분석

- **수행 내용:** 숫자형(numerical) 컬럼들의 핵심 통계량(평균, 표준편차, 사분위수 등)을 확인합니다.
- **선택 근거:** 데이터의 분포를 수치적으로 이해하고, 향후 스케일링이나 이상치 제거 등의 전처리 필요성을 판단하는 기준이 됩니다.

In [ ]:
print('--- 숫자형 데이터 기술 통계 ---')
print(df.describe())

### 1.3. 범주형 데이터 분석

- **수행 내용:** 범주형(categorical) 컬럼들의 고유값(unique value)과 그 빈도를 확인합니다.
- **선택 근거:** 범주형 데이터의 분포를 파악하여 특정 카테고리에 데이터가 편중되어 있는지 확인하고, 카디널리티(cardinality)가 높은 피처의 처리 방안을 고민하는 근거를 마련합니다.

In [ ]:
categorical_features = ['neighbourhood_group', 'neighbourhood', 'room_type']

for feature in categorical_features:
    print(f'--- {feature} 빈도수 ---')
    print(df[feature].value_counts())
    print('
')

## 2단계: 데이터 전처리

EDA 단계에서 파악한 문제점들을 해결하고, 모델링에 적합한 형태로 데이터를 가공합니다.

### 2.1. 결측치 처리

- **수행 내용:** `reviews_per_month`의 결측치는 0으로 채우고, 예측 기여도가 낮을 것으로 보이는 `last_review` 컬럼은 삭제합니다.
- **선택 근거:** 모델은 결측치를 처리할 수 없으므로 반드시 처리해야 합니다. 각 컬럼의 의미와 데이터 생성 맥락을 고려하여 논리적으로 가장 타당한 방식으로 결측치를 처리합니다.

In [ ]:
# reviews_per_month 결측치를 0으로 채우기
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

# last_review 컬럼 삭제
df.drop('last_review', axis=1, inplace=True)

# 결측치 처리 후 확인
print('--- 결측치 처리 후 확인 ---')
print(df.isnull().sum())

### 2.2. 이상치 처리 및 타겟 변수 변환

- **수행 내용:** 가격(`price`)이 0인 데이터는 이상치로 간주하여 제거하고, 오른쪽으로 왜곡된 `price` 분포를 완화하기 위해 로그 변환을 적용합니다.
- **선택 근거:** 가격이 0인 숙소는 분석에 노이즈가 될 수 있습니다. 로그 변환은 이상치의 영향을 줄이고 모델의 예측 안정성을 높이는 데 효과적입니다.

In [ ]:
# price가 0인 데이터 제거
df = df[df['price'] > 0]

# price에 로그 변환 적용 (numpy.log1p는 0을 포함한 데이터에 log(1+x)를 적용하여 변환)
df['price'] = np.log1p(df['price'])

print('--- 로그 변환 후 price 샘플 ---')
print(df['price'].head())

### 2.3. 피처 엔지니어링

- **수행 내용:** 모델 학습에 불필요한 피처를 제거하고, 범주형 피처를 원-핫 인코딩하여 수치형으로 변환합니다.
- **선택 근거:** 모델은 수치 데이터를 입력으로 받으므로 범주형 데이터의 인코딩은 필수적입니다. 예측에 도움이 되지 않는 피처를 제거하여 모델을 더 가볍고 빠르게 만듭니다.

In [ ]:
# 불필요한 피처 제거
df.drop(['id', 'name', 'host_id', 'host_name', 'neighbourhood'], axis=1, inplace=True)

# 원-핫 인코딩 적용
df = pd.get_dummies(df, columns=['neighbourhood_group', 'room_type'], drop_first=True)

print('--- 피처 엔지니어링 후 데이터 샘플 ---')
print(df.head())

## 3단계: 데이터 시각화

### 3.1. 라이브러리 임포트 및 설정

- **수행 내용:** 시각화 라이브러리인 Matplotlib와 Seaborn을 임포트하고 기본 설정을 적용합니다.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 시각화 스타일 설정
plt.style.use('ggplot')

# 한글 폰트 설정 (Windows 기준, 폰트가 없는 경우 설치 필요)
# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['axes.unicode_minus'] = False # 마이너스 부호 깨짐 방지

### 3.2. 타겟 변수(`price`) 분포 시각화

- **수행 내용:** 로그 변환된 `price` 컬럼의 분포를 히스토그램으로 시각화합니다.
- **선택 근거:** 타겟 변수의 분포가 정규성에 가까운지 시각적으로 확인하여, 선형 모델 가정에 부합하는지 검토합니다.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['price'], kde=True)
plt.title('Log Transformed Price Distribution')
plt.show()

### 3.3. 주요 피처와 타겟 변수 간 관계 시각화

- **수행 내용:** 주요 숫자형 피처와 `price` 간의 관계를 산점도로 시각화합니다.
- **선택 근거:** 어떤 피처가 가격에 영향을 미치는지 시각적으로 탐색하고, 피처와 타겟 간의 선형/비선형 관계를 파악합니다.

In [ ]:
numerical_features_for_plot = ['latitude', 'longitude', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

for feature in numerical_features_for_plot:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(x=feature, y='price', data=df, alpha=0.1)
    plt.title(f'{feature} vs Price')
    plt.show()

### 3.4. 피처 간 상관관계 분석

- **수행 내용:** 숫자형 피처들 간의 상관관계를 히트맵으로 시각화합니다.
- **선택 근거:** 피처 간 높은 상관관계(다중공선성)는 선형 모델의 성능을 저해할 수 있습니다. 히트맵을 통해 이러한 관계를 한눈에 파악하고, 필요시 피처 선택의 근거로 활용합니다.

In [ ]:
plt.figure(figsize=(12, 10))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='[.2f]', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

## 4단계: 가설 검정

EDA와 시각화를 통해 세운 잠정적인 가설을 통계적으로 검증합니다.

### 4.1. 가설 검정 수행

- **가설:** "맨해튼(`Manhattan`) 지역의 숙소 평균 가격은 브루클린(`Brooklyn`) 지역의 숙소 평균 가격보다 유의미하게 높을 것이다."
- **수행 내용:** 독립표본 T-검정(Independent Samples T-test)을 수행하여 두 그룹 간의 평균 가격 차이가 통계적으로 유의미한지 검증합니다.
- **선택 근거:** T-검정은 두 집단의 평균 차이가 통계적으로 유의미한지를 검증하는 대표적인 통계 분석 방법입니다.

In [ ]:
from scipy import stats

# 원본 데이터에서 맨해튼과 브루클린 데이터 필터링 (로그 변환 전 가격으로 비교)
original_df = pd.read_csv('AB_NYC_2019.csv')
manhattan = original_df[(original_df['neighbourhood_group'] == 'Manhattan') & (original_df['price'] > 0)]['price']
brooklyn = original_df[(original_df['neighbourhood_group'] == 'Brooklyn') & (original_df['price'] > 0)]['price']

# 독립표본 T-검정 수행
t_stat, p_value = stats.ttest_ind(manhattan, brooklyn, equal_var=False) # 등분산성 가정하지 않음 (Welch's t-test)

print(f'T-statistic: {t_stat}')
print(f'P-value: {p_value}')

alpha = 0.05
if p_value < alpha:
    print('결론: 귀무가설 기각. 맨해튼과 브루클린의 평균 숙소 가격에는 통계적으로 유의미한 차이가 있습니다.')
else:
    print('결론: 귀무가설 채택. 맨해튼과 브루클린의 평균 숙소 가격 차이는 통계적으로 유의미하지 않습니다.')

## 5단계: 모델링

전처리가 완료된 데이터를 사용하여 가격 예측 모델을 구축하고 평가합니다.

### 5.1. 데이터 분할

- **수행 내용:** 피처(X)와 타겟(y, `price`)을 분리하고, 전체 데이터를 훈련(train) 데이터와 테스트(test) 데이터로 분할합니다.
- **선택 근거:** 모델이 보지 못한 새로운 데이터(테스트 데이터)로 성능을 평가해야 일반화 성능을 객관적으로 측정할 수 있습니다.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train data shape: {X_train.shape}')
print(f'Test data shape: {X_test.shape}')

### 5.2. 모델 선택 및 학습

- **수행 내용:** 기준 모델인 선형 회귀와, 더 복잡한 모델인 랜덤 포레스트 회귀 모델을 학습시킵니다.
- **선택 근거:** 단순한 모델부터 복잡한 모델까지 다양하게 시도하여 문제에 가장 적합한 모델을 찾습니다. 선형 회귀로 성능의 기준선을 설정하고, 랜덤 포레스트로 성능 향상을 시도합니다.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# 선형 회귀 모델 학습
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# 랜덤 포레스트 모델 학습
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

print('--- 모델 학습 완료 ---')

### 5.3. 모델 평가

- **수행 내용:** 테스트 데이터에 대한 두 모델의 예측 성능을 MAE, MSE, R² 지표로 평가합니다. 로그 변환된 타겟을 원래 스케일로 되돌려 평가를 수행합니다.
- **선택 근거:** 각 평가 지표는 모델의 성능을 다른 관점에서 보여주므로, 종합적으로 판단해야 합니다.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 예측 수행
y_pred_lr = lr_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)

# 로그 변환된 값을 원래 스케일로 되돌리기 (np.expm1은 log1p의 역함수)
y_test_orig = np.expm1(y_test)
y_pred_lr_orig = np.expm1(y_pred_lr)
y_pred_rf_orig = np.expm1(y_pred_rf)

def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'--- {model_name} 평가 결과 ---')
    print(f'MAE: {mae:.2f}')
    print(f'MSE: {mse:.2f}')
    print(f'R²: {r2:.2f}')
    print('
')

evaluate_model(y_test_orig, y_pred_lr_orig, 'Linear Regression')
evaluate_model(y_test_orig, y_pred_rf_orig, 'Random Forest')

### 5.4. 하이퍼파라미터 튜닝 (예시)

- **수행 내용:** 랜덤 포레스트 모델의 성능을 최적화하기 위해 GridSearchCV를 사용하여 최적의 하이퍼파라미터를 탐색하는 방법을 예시로 보여줍니다.
- **선택 근거:** 하이퍼파라미터 튜닝은 모델의 잠재 성능을 최대한으로 이끌어내기 위한 필수적인 과정입니다.

**참고:** 아래 코드는 실행에 많은 시간이 소요될 수 있습니다. 실제 분석 시에는 파라미터 범위를 조정하거나 `RandomizedSearchCV`를 사용하는 것이 효율적일 수 있습니다.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 탐색할 파라미터 그리드 설정
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

# GridSearchCV 객체 생성
grid_search = GridSearchCV(estimator=RandomForestRegressor(random_state=42, n_jobs=-1), 
                           param_grid=param_grid, 
                           cv=3, 
                           scoring='neg_mean_squared_error', 
                           verbose=2)

# 아래 .fit() 라인의 주석을 해제하여 그리드 서치를 실행할 수 있습니다.
# grid_search.fit(X_train, y_train)

# print(f'Best parameters found: {grid_search.best_params_}')

print('--- 하이퍼파라미터 튜닝 코드 준비 완료 ---')